# 04 — Path A: Corpus diversification (paradigm verdict)

**Hypothesis (사용자)**: 다양한 cognitive task corpus 가 expert specialization 의 emergence 의 필요조건.

**추가된 source (data.py revision 5)**:
- **ROCStories** (내러티브 / storytelling) — ~30k stories
- **αNLI** (`allenai/art`, abductive reasoning) — ~20k
- **SocialIQA** (theory of mind / social commonsense) — ~20k

총 corpus = 기존 (Pennebaker + Reddit + PANDORA + PersonaChat ~118k) + 새 70k ≈ **190k samples**.

**Verdict**:
- Expert 분화 명확 (각 expert top-keyword 가 cognitive op 별 coherent) → **corpus 가 primary**. paradigm 살아남.
- 현재 와 같은 disguised collapse → **architecture/objective 가 primary**. Option II (top-k + weak supervision) 진행.

**Pre-req**: 00_setup 먼저 실행. data.py 변경 업로드 타이밍 주의.

In [ ]:
# Pre-setup
import os, sys
BASE = '/content/drive/MyDrive/sideproject'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)
print('cwd =', os.getcwd())

## A. 새 corpus + BGE encode + cycle pretrain (~3-4시간)

data.py 의 `cache_key` 가 새 source flag 포함 → 새 corpus parquet + 새 fact_emb npy 자동 생성.

- BGE encode: ~2-3시간 (190k samples on L4)
- Cycle pretrain: ~1시간 (epoch 30, batch 32)

Auto-resume 작동 — session timeout 시 같은 cell 재실행.

In [ ]:
from phase1.train import train_phase1

train_phase1(
    run_id='ph1_v4_diverse',
    epochs=30,
    batch_size=32,
    lr=1e-4,
    warmup_steps=300,
    grad_clip=1.0,
    lambda_lb=0.1,
    lambda_ortho=0.0,
)

## B. History + eval (cycle reconstruction + linear probe)

ph1_v3_minimal 값과 비교:
- ph1_v3_minimal: mean_cosine=0.8777, alpha macro_f1=0.4942, K_active=13.93
- ph1_v4_diverse: 아래 제출

In [ ]:
import json
from pathlib import Path
from phase1.eval import eval_phase1

run_id = 'ph1_v4_diverse'
h = json.loads(Path(f'out/phase1/{run_id}/history.json').read_text())
first, last = h[0], h[-1]
print(f'[{run_id}] epoch {first["epoch"]} \u2192 {last["epoch"]}')
print(f'  loss        {first["loss"]:.4f} \u2192 {last["loss"]:.4f}')
print(f'  loss_recon  {first["loss_recon"]:.4f} \u2192 {last["loss_recon"]:.4f}')
print(f'  K_active    {first["active_frac"]*16:.2f} \u2192 {last["active_frac"]*16:.2f}  (of 16)')

result = eval_phase1(run_id=run_id)
print('\n=== summary ===')
print(f'mean_cosine: {result["cycle_reconstruction"]["mean_cosine"]:.4f}')
print(f'alpha macro_f1: {result["linear_probe_routed_alpha"].get("macro_f1"):.4f}')
print(f'subkg macro_f1: {result["linear_probe_sub_kg"].get("macro_f1"):.4f}')

## C. Cluster analysis — paradigm verdict (★ 진짜 판정)

각 expert 의:
- n_active 샘플 수 (alpha > 0.01)
- top-100 activating sample 의 source distribution
- top-100 activating sample 의 TF-IDF top keyword

**Coherent specialization 관찰 되면** (e.g. Expert 3 = narrative, Expert 7 = abductive, Expert 11 = social, Expert 14 = factual) → corpus diversification 이 paradigm 의 emergence 조건.

**현재 와 같은 random keyword** → architecture/objective 가 primary.

In [ ]:
from phase1.cluster_analysis import analyze_experts

analyze_experts('ph1_v4_diverse')

## D. SimBench eval — downstream performance 도 함께 확인

Cluster verdict 와 별개로, SimBench classifier head 학습 → test_acc 비교:
- ph1_v3_minimal: 0.7120
- ph1_v4_diverse: ?

Corpus diversification 이 downstream perf 에도 도움 주는지 (orthogonal verification).

In [ ]:
from phase1.eval_simbench_classifier import train_simbench_classifier

train_simbench_classifier(
    run_id='ph1_v4_diverse',
    epochs=100,
    batch_size=256,
    lr=1e-3,
    hidden=512,
    dropout=0.1,
)

## E. 최종 비교 — ph1_v3_minimal vs ph1_v4_diverse

In [ ]:
from phase1.cluster_analysis import compare_runs

compare_runs(['ph1_v3_minimal', 'ph1_v4_diverse'])